In [ ]:
import os
import re
import json
import time
import warnings
from pathlib import Path
from itertools import combinations
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
load_dotenv()

# -----------------------------
# Global configuration
# -----------------------------
CSV_PATH = "cleaned_caption_Derm1M.csv"
SCHEMA_OUT = "feature_schema.json"
PHASE2_OUT = "derm1m_features.csv"
SCIN_CSV = "SCIN-dataset/dataset_scin_cases.csv"

DISCOVERY_DIR = Path("discovery_outputs")
CHECKPOINT_DIR = Path("checkpoints")
ANALYSIS_DIR = Path("analysis_outputs")
COOC_DIR = ANALYSIS_DIR / "cooccurrence"

for d in [DISCOVERY_DIR, CHECKPOINT_DIR, ANALYSIS_DIR, COOC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# LLM backend selection
# -----------------------------
# "api"  : OpenAI-compatible endpoint that serves Qwen3-8B
# "local": transformers AutoModelForCausalLM (Qwen/Qwen3-8B)
LLM_MODE = "api"  # change to "local" if running local model

QWEN_MODEL_ID = "Qwen/Qwen3-8B"

# API mode config
OPENAI_COMPAT_BASE_URL = os.getenv("OPENAI_COMPAT_BASE_URL", "")  # e.g. http://localhost:8000/v1
OPENAI_COMPAT_API_KEY = os.getenv("OPENAI_COMPAT_API_KEY", "")

# Local mode config
LOCAL_DEVICE_MAP = "auto"
LOCAL_TORCH_DTYPE = "auto"

ENABLE_THINKING = True
GENERATION_CFG = {
    # Qwen3 recommended defaults (thinking mode)
    "temperature": 0.6 if ENABLE_THINKING else 0.7,
    "top_p": 0.95 if ENABLE_THINKING else 0.8,
    "top_k": 20,
    "max_new_tokens": 4096,
}

print(f"LLM_MODE = {LLM_MODE}")



In [ ]:
class QwenLLM:
    """Unified interface for Qwen3-8B via API or local transformers."""

    def __init__(
        self,
        mode: str,
        model_id: str = QWEN_MODEL_ID,
        base_url: str = "",
        api_key: str = "",
        enable_thinking: bool = True,
        generation_cfg: dict | None = None,
    ):
        self.mode = mode
        self.model_id = model_id
        self.base_url = base_url.rstrip("/") if base_url else ""
        self.api_key = api_key
        self.enable_thinking = enable_thinking
        self.generation_cfg = generation_cfg or {}

        self.tokenizer = None
        self.model = None

        if self.mode == "local":
            from transformers import AutoTokenizer, AutoModelForCausalLM
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id,
                torch_dtype=LOCAL_TORCH_DTYPE,
                device_map=LOCAL_DEVICE_MAP,
            )

        elif self.mode == "api":
            if not self.base_url:
                raise ValueError("Set OPENAI_COMPAT_BASE_URL for API mode")
            if not self.api_key:
                raise ValueError("Set OPENAI_COMPAT_API_KEY for API mode")
        else:
            raise ValueError("mode must be 'api' or 'local'")

    @staticmethod
    def _strip_code_fences(text: str) -> str:
        return re.sub(r"```json\s*|```\s*", "", text).strip()

    @staticmethod
    def _split_think_content(text: str) -> str:
        # Keep only final answer if model emits <think>...</think>
        return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    def _api_generate(self, prompt: str) -> str:
        url = f"{self.base_url}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": self.model_id,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": self.generation_cfg.get("temperature", 0.6),
            "top_p": self.generation_cfg.get("top_p", 0.95),
            "max_tokens": self.generation_cfg.get("max_new_tokens", 4096),
        }
        r = requests.post(url, headers=headers, json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        txt = data["choices"][0]["message"]["content"]
        return txt

    def _local_generate(self, prompt: str) -> str:
        text = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=self.enable_thinking,
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.generation_cfg.get("max_new_tokens", 4096),
            temperature=self.generation_cfg.get("temperature", 0.6),
            top_p=self.generation_cfg.get("top_p", 0.95),
            do_sample=True,
        )
        out_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        return self.tokenizer.decode(out_ids, skip_special_tokens=True)

    def generate_text(self, prompt: str, retries: int = 3) -> str:
        for attempt in range(retries):
            try:
                if self.mode == "api":
                    text = self._api_generate(prompt)
                else:
                    text = self._local_generate(prompt)
                text = self._split_think_content(text)
                return text.strip()
            except Exception as e:
                if attempt == retries - 1:
                    raise
                sleep_s = 2 ** attempt
                print(f"LLM error: {e}. Retrying in {sleep_s}s...")
                time.sleep(sleep_s)
        return ""

    def generate_json(self, prompt: str, retries: int = 3):
        for attempt in range(retries):
            try:
                text = self.generate_text(prompt, retries=1)
                text = self._strip_code_fences(text)
                return json.loads(text)
            except Exception as e:
                if attempt == retries - 1:
                    raise
                sleep_s = 2 ** attempt
                print(f"JSON parse/generation error: {e}. Retrying in {sleep_s}s...")
                time.sleep(sleep_s)


llm = QwenLLM(
    mode=LLM_MODE,
    model_id=QWEN_MODEL_ID,
    base_url=OPENAI_COMPAT_BASE_URL,
    api_key=OPENAI_COMPAT_API_KEY,
    enable_thinking=ENABLE_THINKING,
    generation_cfg=GENERATION_CFG,
)
print("Qwen client initialized")



In [ ]:
# =====================================================
# PHASE 1: Feature Schema Discovery (Qwen3-8B)
# =====================================================

BATCH_SIZE = 20

SAMPLING_TIERS = [
    (5000, 200),
    (1000, 150),
    (200, 100),
    (0, None),
]

def compute_sample_size(class_count: int) -> int:
    for threshold, n_sample in SAMPLING_TIERS:
        if class_count > threshold:
            return n_sample if n_sample is not None else class_count
    return class_count


def load_and_sample(csv_path: str) -> pd.DataFrame:
    print("Loading full CSV...")
    df = pd.read_csv(csv_path)
    df["truncated_caption"] = df["truncated_caption"].fillna("")
    df = df[df["truncated_caption"].str.strip().ne("")].copy()

    print(f"  {len(df):,} records with non-empty captions")
    print(f"  {df['label_name'].nunique()} unique label_names")
    print(f"  {df['disease_label'].nunique()} unique disease_labels\n")

    label_counts = df["label_name"].value_counts()
    sampled_parts = []

    for label_name, total_count in label_counts.items():
        n_sample = min(compute_sample_size(total_count), total_count)
        subset = df[df["label_name"] == label_name]
        n_sublabels = subset["disease_label"].nunique()

        if n_sublabels > 1 and n_sample < total_count:
            per_sub = max(1, n_sample // n_sublabels)
            stratified = (
                subset.groupby("disease_label", group_keys=False)
                .apply(lambda g: g.sample(min(len(g), per_sub), random_state=42))
            )
            if len(stratified) > n_sample:
                stratified = stratified.sample(n_sample, random_state=42)
            sampled_parts.append(stratified)
        else:
            sampled_parts.append(subset.sample(n_sample, random_state=42))

    sampled = pd.concat(sampled_parts, ignore_index=True)
    print(f"  Sampled {len(sampled):,} captions across {sampled['label_name'].nunique()} label_names")
    return sampled


DISCOVERY_PROMPT = """You are a clinical NLP expert specialising in dermatology.

Given the following batch of skin disease case captions (all belonging to the
disease category \"{label_name}\"), extract ALL possible clinical and contextual
feature categories that are ACTUALLY MENTIONED or DESCRIBED in these captions.

IMPORTANT RULES:
- Extract ONLY information that is ACTUALLY STATED in these captions.
- Do NOT infer or assume features that are not mentioned.
- Separate compound features (e.g. red scaly lesion -> color=red, texture=scaly)
- Keep features medically meaningful and distinguishable.

Return ONLY valid JSON (no markdown) in this structure:
{
  "feature_categories": [
    {
      "name": "snake_case_feature_name",
      "category": "demographics | morphology | body_location | symptoms | duration | triggers | treatments | clinical_signs | history | severity | other",
      "description": "brief clinical description",
      "example_values": ["value1", "value2", "value3"],
      "is_binary": true,
      "extraction_examples": ["phrase from caption"]
    }
  ]
}
"""


def discover_features_batch(captions: list[str], label_name: str, retries: int = 3) -> list[dict]:
    numbered = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(captions))
    prompt = DISCOVERY_PROMPT.format(label_name=label_name) + f"\n\nHere are {len(captions)} captions for '{label_name}':\n\n{numbered}"

    for attempt in range(retries):
        try:
            parsed = llm.generate_json(prompt, retries=1)
            if isinstance(parsed, dict):
                return parsed.get("feature_categories", [])
            if isinstance(parsed, list):
                return parsed
        except Exception as e:
            print(f"    Discovery error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)
    return []


def discover_all_features(sample_df: pd.DataFrame) -> dict[str, dict]:
    all_features: dict[str, dict] = {}
    label_names = sample_df["label_name"].unique()

    for label_name in tqdm(label_names, desc="Discovering features per label"):
        label_captions = sample_df[sample_df["label_name"] == label_name]["truncated_caption"].tolist()
        safe_name = re.sub(r"[^\w\-]", "_", str(label_name))[:60]
        cache_path = DISCOVERY_DIR / f"discovery_{safe_name}.json"

        if cache_path.exists():
            with open(cache_path, "r", encoding="utf-8") as f:
                label_features = json.load(f)
        else:
            label_features = []
            for start in range(0, len(label_captions), BATCH_SIZE):
                batch = label_captions[start:start + BATCH_SIZE]
                label_features.extend(discover_features_batch(batch, label_name))
                time.sleep(0.5)
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(label_features, f, indent=2)

        for feat in label_features:
            name = feat.get("name", "").lower().strip().replace(" ", "_")
            if not name:
                continue
            if name not in all_features:
                all_features[name] = feat
                all_features[name]["found_in_labels"] = [label_name]
            else:
                existing_ex = set(all_features[name].get("example_values", []))
                new_ex = set(feat.get("example_values", []))
                all_features[name]["example_values"] = list(existing_ex | new_ex)[:10]

                existing_extr = set(all_features[name].get("extraction_examples", []))
                new_extr = set(feat.get("extraction_examples", []))
                all_features[name]["extraction_examples"] = list(existing_extr | new_extr)[:5]

                if label_name not in all_features[name].get("found_in_labels", []):
                    all_features[name]["found_in_labels"].append(label_name)

    return all_features


CONSOLIDATION_PROMPT = """You are a senior clinical NLP engineer building a canonical
feature schema for a skin disease classification system.

You are given a raw list of {n_features} clinical feature categories extracted
across {n_disease_classes} disease classes.

Tasks:
1. Deduplicate synonyms and merge overlaps.
2. Standardize names to snake_case.
3. Keep useful differential-diagnosis features.
4. For each feature output is_binary true/false.
5. Set regex_extractable true/false.

Return ONLY valid JSON:
{
  "feature_categories": [
    {
      "name": "snake_case_canonical_name",
      "category": "demographics | morphology | body_location | symptoms | duration | triggers | treatments | clinical_signs | history | severity | other",
      "description": "brief clinical description",
      "example_values": ["val1", "val2"],
      "is_binary": true,
      "regex_extractable": false
    }
  ]
}

Raw feature list:
{raw_features}
"""


def consolidate_schema(all_features: dict[str, dict], n_captions_sampled: int = 0, n_disease_classes: int = 0) -> dict:
    compact = []
    for name, feat in all_features.items():
        compact.append({
            "name": name,
            "category": feat.get("category", "other"),
            "description": feat.get("description", ""),
            "example_values": feat.get("example_values", [])[:5],
            "is_binary": feat.get("is_binary", True),
            "n_labels_found": len(feat.get("found_in_labels", [])),
        })

    prompt = CONSOLIDATION_PROMPT.format(
        n_features=len(compact),
        n_captions_sampled=n_captions_sampled,
        n_disease_classes=n_disease_classes,
        raw_features=json.dumps(compact, indent=1),
    )

    for attempt in range(3):
        try:
            parsed = llm.generate_json(prompt, retries=1)
            if isinstance(parsed, list):
                parsed = {"feature_categories": parsed}
            if "feature_categories" in parsed:
                return parsed
        except Exception as e:
            print(f"  Consolidation error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)

    return {"feature_categories": compact}


SCIN_FEATURES = {
    "textures_raised_or_bumpy": "texture_raised",
    "textures_flat": "texture_flat",
    "textures_rough_or_flaky": "texture_rough_flaky",
    "textures_fluid_filled": "texture_fluid_filled",
    "condition_symptoms_itching": "symptom_itching",
    "condition_symptoms_burning": "symptom_burning",
    "condition_symptoms_pain": "symptom_pain",
    "condition_symptoms_bleeding": "symptom_bleeding",
    "condition_symptoms_increasing_size": "symptom_increasing_size",
    "condition_symptoms_darkening": "symptom_darkening",
    "condition_symptoms_bothersome_appearance": "symptom_bothersome_appearance",
    "other_symptoms_fever": "symptom_fever",
    "other_symptoms_chills": "symptom_chills",
    "other_symptoms_fatigue": "symptom_fatigue",
    "other_symptoms_joint_pain": "symptom_joint_pain",
    "other_symptoms_mouth_sores": "symptom_mouth_sores",
    "other_symptoms_shortness_of_breath": "symptom_shortness_of_breath",
    "body_parts_head_or_neck": "location_head_neck",
    "body_parts_arm": "location_arm",
    "body_parts_palm": "location_palm",
    "body_parts_back_of_hand": "location_back_of_hand",
    "body_parts_torso_front": "location_torso_front",
    "body_parts_torso_back": "location_torso_back",
    "body_parts_genitalia_or_groin": "location_genitalia_groin",
    "body_parts_buttocks": "location_buttocks",
    "body_parts_leg": "location_leg",
    "body_parts_foot_top_or_side": "location_foot_top_side",
    "body_parts_foot_sole": "location_foot_sole",
    "age_group": "age_group",
    "sex_at_birth": "sex",
    "fitzpatrick_skin_type": "fitzpatrick_skin_type",
    "condition_duration": "duration",
}

def _infer_category(name: str) -> str:
    if name.startswith("symptom_"): return "symptoms"
    if name.startswith("location_"): return "body_location"
    if name.startswith("texture_") or name.startswith("color_"): return "morphology"
    if name.startswith("trigger_"): return "triggers"
    if name.startswith("treatment_"): return "treatments"
    if name in ("age_group", "sex", "fitzpatrick_skin_type"): return "demographics"
    if name == "duration": return "duration"
    return "other"

def _is_regex_extractable(name: str) -> bool:
    regex_prefixes = ("symptom_", "location_", "texture_", "color_", "distribution_")
    regex_names = {"age_group", "sex", "fitzpatrick_skin_type", "duration", "onset_sudden", "lesion_count", "diagnosis_confidence"}
    return name.startswith(regex_prefixes) or name in regex_names

def add_scin_alignment(schema: dict) -> dict:
    for scin_col, canonical in SCIN_FEATURES.items():
        matched = next((f for f in schema["feature_categories"] if f.get("name") == canonical), None)
        if matched:
            matched["scin_column"] = scin_col
            matched["scin_comparable"] = True
        else:
            schema["feature_categories"].append({
                "name": canonical,
                "category": _infer_category(canonical),
                "description": f"Mapped from SCIN column: {scin_col}",
                "example_values": [],
                "is_binary": True,
                "regex_extractable": _is_regex_extractable(canonical),
                "scin_column": scin_col,
                "scin_comparable": True,
                "derm1m_sourced": False,
            })
    return schema


def run_phase1(csv_path: str = CSV_PATH, schema_out: str = SCHEMA_OUT) -> dict:
    print("=" * 60)
    print("  PHASE 1: Feature Schema Discovery (Bottom-Up)")
    print("=" * 60 + "\n")

    sample_df = load_and_sample(csv_path)
    sample_path = DISCOVERY_DIR / "sampled_captions.csv"
    sample_df.to_csv(sample_path, index=False)
    print(f"Saved sample: {sample_path}")

    all_features = discover_all_features(sample_df)
    print(f"Raw features discovered: {len(all_features)}")

    raw_path = DISCOVERY_DIR / "raw_features_all.json"
    raw_serializable = {k: dict(v) for k, v in all_features.items()}
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(raw_serializable, f, indent=2)

    schema = consolidate_schema(
        all_features,
        n_captions_sampled=len(sample_df),
        n_disease_classes=int(sample_df["label_name"].nunique()),
    )
    schema = add_scin_alignment(schema)

    n_total = len(schema["feature_categories"])
    n_comparable = sum(1 for f in schema["feature_categories"] if f.get("scin_comparable"))
    n_regex = sum(1 for f in schema["feature_categories"] if f.get("regex_extractable"))

    schema["metadata"] = {
        "source_csv": csv_path,
        "n_captions_sampled": len(sample_df),
        "n_label_names": int(sample_df["label_name"].nunique()),
        "n_total_features": n_total,
        "n_scin_comparable": n_comparable,
        "n_regex_extractable": n_regex,
        "n_llm_only": n_total - n_regex,
        "model_used": QWEN_MODEL_ID,
        "llm_mode": LLM_MODE,
    }

    with open(schema_out, "w", encoding="utf-8") as fp:
        json.dump(schema, fp, indent=2)

    print(f"Schema saved to: {schema_out}")
    return schema



In [ ]:
# =====================================================
# PHASE 2: Bulk Feature Extraction (Hybrid NLP + Qwen)
# =====================================================

LLM_BATCH_SIZE = 30
MAX_CAPTION_LEN = 500


def load_schema(schema_path: str) -> dict:
    with open(schema_path, "r", encoding="utf-8") as f:
        schema = json.load(f)
    print(f"Loaded schema: {len(schema.get('feature_categories', []))} features")
    return schema


def split_features_by_lane(schema: dict) -> tuple[list[dict], list[dict]]:
    regex_feats, llm_feats = [], []
    for feat in schema.get("feature_categories", []):
        (regex_feats if feat.get("regex_extractable", False) else llm_feats).append(feat)
    return regex_feats, llm_feats


AGE_RE = re.compile(r"\b(\d{1,3})[- ]?(?:year|yr)s?[- ]?old\b|\bage[d]?\s*:?\s*(\d{1,3})\b", re.I)
SEX_RE = re.compile(r"\b(male|female|man|woman|boy|girl|he\b|she\b|his\b|her\b)\b", re.I)

def extract_age(text: str) -> tuple:
    m = AGE_RE.search(text)
    if not m:
        return None, "unknown"
    age = int(m.group(1) or m.group(2))
    if age < 13: return age, "child"
    if age < 18: return age, "adolescent"
    if age < 30: return age, "18_29"
    if age < 45: return age, "30_44"
    if age < 60: return age, "45_59"
    if age < 75: return age, "60_74"
    return age, "75_plus"


def extract_sex(text: str) -> int:
    m = SEX_RE.search(text)
    if not m:
        return 2
    w = m.group(1).lower()
    if w in ("male", "man", "boy", "he", "his"): return 1
    if w in ("female", "woman", "girl", "she", "her"): return 0
    return 2


def extract_fitzpatrick(text: str) -> int:
    m = re.search(r"fitzpatrick\s*(type|skin)?\s*(\d)", text, re.I)
    if m:
        return int(m.group(2))
    lower = text.lower()
    if re.search(r"fair|type [i1]\b|very light", lower): return 1
    if re.search(r"light|type [ii2]\b", lower): return 2
    if re.search(r"medium|olive|type [iii3]\b", lower): return 3
    if re.search(r"brown|type [iv4]\b", lower): return 4
    if re.search(r"dark brown|type [v5]\b", lower): return 5
    if re.search(r"deeply pigmented|type [vi6]\b", lower): return 6
    return 0


LOCATION_PATTERNS = {
    "location_face": ["face", "cheek", "forehead", "nose", "chin", "perioral", "periorbital", "eyelid", "lip"],
    "location_scalp": ["scalp", "hair", "hairline"],
    "location_head_neck": ["head", "neck", "cervical"],
    "location_trunk": ["trunk", "torso", "chest", "abdomen", "back", "flank"],
    "location_torso_front": ["chest", "abdomen", "torso front", "anterior trunk"],
    "location_torso_back": ["back", "posterior trunk", "dorsal"],
    "location_arm": ["arm", "forearm", "elbow", "upper arm", "antecubital"],
    "location_hand": ["hand", "finger", "knuckle", "wrist", "dorsum of hand"],
    "location_palm": ["palm", "palmar"],
    "location_back_of_hand": ["back of hand", "dorsum of hand"],
    "location_leg": ["leg", "thigh", "shin", "calf", "popliteal", "knee"],
    "location_foot": ["foot", "feet", "toe", "ankle"],
    "location_foot_top_side": ["dorsum of foot", "top of foot"],
    "location_foot_sole": ["sole", "plantar", "heel"],
    "location_genitalia_groin": ["genital", "groin", "inguinal", "scrotal", "vulva", "perineal", "perianal"],
    "location_buttocks": ["buttock", "gluteal"],
    "location_mouth": ["mouth", "oral", "mucosa", "tongue", "buccal", "gum"],
    "location_nail": ["nail", "ungual", "periungual", "subungual"],
    "location_axilla": ["axilla", "axillary", "armpit"],
    "location_widespread": ["widespread", "generalised", "generalized", "whole body", "diffuse", "all over"],
}

def extract_locations(text: str) -> dict[str, int]:
    lower = text.lower()
    return {feat: (1 if any(kw in lower for kw in kws) else 2) for feat, kws in LOCATION_PATTERNS.items()}


TEXTURE_PATTERNS = {
    "texture_raised": ["raised", "elevated", "papule", "nodule", "plaque", "bump", "boil", "wart", "verruca", "papular"],
    "texture_flat": ["flat", "macular", "macule", "patch"],
    "texture_rough_flaky": ["rough", "scaly", "scale", "flak", "desquam", "keratotic", "hyperkeratotic"],
    "texture_fluid_filled": ["vesicle", "blister", "bulla", "pustule", "fluid", "weeping", "oozing"],
    "texture_ulcerated": ["ulcer", "ulcerat", "erosion", "erode", "crusted", "crust", "excoriat"],
    "texture_smooth": ["smooth", "shiny", "glossy"],
}

COLOR_PATTERNS = {
    "color_red": ["red", "erythema", "erythematous", "pink", "violaceous", "purpura", "petechiae"],
    "color_brown": ["brown", "hyperpigment", "pigment", "tan", "bronze"],
    "color_white": ["white", "hypopigment", "depigment", "pale", "leucoder"],
    "color_yellow": ["yellow", "xanth"],
    "color_black": ["black", "dark", "melanotic", "melanin"],
    "color_blue_grey": ["blue", "grey", "gray", "slate"],
}

DISTRIBUTION_PATTERNS = {
    "distribution_unilateral": ["unilateral", "one side", "left ", "right "],
    "distribution_bilateral": ["bilateral", "both sides", "symmetric"],
    "distribution_dermatomal": ["dermatome", "dermatomal"],
    "distribution_grouped": ["grouped", "cluster", "herpetiform"],
    "distribution_linear": ["linear", "streak", "along"],
    "distribution_annular": ["annular", "ring", "ring-like", "circular"],
}

def extract_morphology(text: str) -> dict[str, int]:
    lower = text.lower()
    result = {}
    all_patterns = {**TEXTURE_PATTERNS, **COLOR_PATTERNS, **DISTRIBUTION_PATTERNS}
    for feat, kws in all_patterns.items():
        result[feat] = 1 if any(kw in lower for kw in kws) else 2
    return result


SYMPTOM_PATTERNS = {
    "symptom_itching": ["itch", "pruritic", "pruritus"],
    "symptom_burning": ["burn", "burning", "stinging"],
    "symptom_pain": ["pain", "painful", "tender", "sore", "hurt"],
    "symptom_bleeding": ["bleed", "hemorrhage", "haemorrhage"],
    "symptom_increasing_size": ["growing", "enlarg", "increas", "spreading", "expand"],
    "symptom_darkening": ["darken", "pigment increas", "getting darker"],
    "symptom_bothersome_appearance": ["bothersome", "unsightly", "cosmetic concern", "disfigur"],
    "symptom_fever": ["fever", "febrile", "pyrexia"],
    "symptom_chills": ["chill", "rigor"],
    "symptom_fatigue": ["fatigue", "tired", "malaise", "lethargy"],
    "symptom_joint_pain": ["joint pain", "arthralgia", "arthrit"],
    "symptom_mouth_sores": ["mouth sore", "oral ulcer", "aphth", "mucosal lesion"],
    "symptom_shortness_of_breath": ["shortness of breath", "dyspnea", "dyspnoea", "breathing difficult"],
    "symptom_nail_change": ["nail change", "onychol", "nail dystrophy", "nail pitting"],
    "symptom_hair_loss": ["hair loss", "alopecia", "bald"],
}

def extract_symptoms(text: str) -> dict[str, int]:
    lower = text.lower()
    return {feat: (1 if any(kw in lower for kw in kws) else 2) for feat, kws in SYMPTOM_PATTERNS.items()}


DURATION_MAP = [
    (r"\b(\d+)\s*hour", "hours"),
    (r"\b(\d+)\s*day", "days"),
    (r"\b(\d+)\s*week", "weeks"),
    (r"\b(\d+)\s*month", "months"),
    (r"\b(\d+)\s*year", "years"),
    (r"since childhood|lifelong|congenital|birth", "lifelong"),
    (r"chronic|long.stand", "chronic"),
    (r"acute|sudden|abrupt", "acute"),
]

def extract_duration(text: str) -> str:
    lower = text.lower()
    for pat, bucket in DURATION_MAP:
        if re.search(pat, lower):
            return bucket
    return "unknown"


def extract_onset(text: str) -> int:
    lower = text.lower()
    if re.search(r"sudden|abrupt|acute onset|overnight|appeared\s+(yesterday|today)", lower):
        return 1
    if re.search(r"gradual|slowly|progressive|over (time|weeks|months|years)", lower):
        return 0
    return 2


def extract_diagnosis_confidence(text: str) -> str:
    lower = text.lower()
    if re.search(r"confirmed|biopsy.proven|histolog|patholog|laboratory", lower): return "confirmed"
    if re.search(r"clinically diagnosed|consistent with|impression", lower): return "clinical"
    if re.search(r"self.assumed|self.diagnos|possible|likely|suspected|probable", lower): return "suspected"
    if re.search(r"no definitive|uncertain|unclear|differential", lower): return "uncertain"
    return "unknown"


def extract_lesion_count(text: str) -> str:
    lower = text.lower()
    if re.search(r"\bsingle\b|solitary|one lesion|isolated", lower): return "single"
    if re.search(r"\bfew\b|several|multiple|numerous|many", lower): return "multiple"
    if re.search(r"diffuse|widespread|generali[sz]ed", lower): return "widespread"
    return "unknown"


def extract_row_rules(caption: str) -> dict:
    age_num, age_bucket = extract_age(caption)
    row = {
        "age_numeric": age_num,
        "age_group": age_bucket,
        "sex": extract_sex(caption),
        "fitzpatrick_skin_type": extract_fitzpatrick(caption),
        "duration_bucket": extract_duration(caption),
        "onset_sudden": extract_onset(caption),
        "diagnosis_confidence": extract_diagnosis_confidence(caption),
        "lesion_count": extract_lesion_count(caption),
    }
    row.update(extract_locations(caption))
    row.update(extract_morphology(caption))
    row.update(extract_symptoms(caption))
    return row


def build_llm_feature_list(llm_feats: list[dict]) -> list[str]:
    return [f["name"] for f in llm_feats]


def build_llm_system_prompt(llm_feature_names: list[str]) -> str:
    feature_list_str = json.dumps(llm_feature_names, indent=2)
    return f"""You are a clinical dermatology NLP specialist.
Extract the following features from each skin disease caption.
Return ONLY a valid JSON array — one object per caption — with these exact keys:
{feature_list_str}

ENCODING RULES:
- Binary features:
  1 = present/mentioned
  0 = explicitly absent
  2 = unknown/not mentioned
- Categorical features:
  output best value string, or \"unknown\".

IMPORTANT:
- Extract only explicit information; do not infer.
- No markdown fences, no prose.
"""


def extract_llm_features(captions: list[str], llm_feature_names: list[str], system_prompt: str, retries: int = 3) -> list[dict]:
    truncated = [c[:MAX_CAPTION_LEN] for c in captions]
    numbered = "\n\n".join(f"[{i}] {c}" for i, c in enumerate(truncated))
    full_prompt = system_prompt + "\n\n" + numbered

    for attempt in range(retries):
        try:
            parsed = llm.generate_json(full_prompt, retries=1)
            if isinstance(parsed, list) and len(parsed) == len(captions):
                return parsed
            if isinstance(parsed, list) and len(parsed) > 0:
                if len(parsed) < len(captions):
                    fallback = {k: 2 for k in llm_feature_names}
                    parsed.extend([fallback] * (len(captions) - len(parsed)))
                return parsed[:len(captions)]
        except Exception as e:
            print(f"    LLM batch error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)

    return [{k: 2 for k in llm_feature_names} for _ in captions]


def run_phase2(csv_path: str = CSV_PATH, schema_path: str = SCHEMA_OUT, output_csv: str = PHASE2_OUT, use_llm: bool = True):
    print("=" * 60)
    print("  PHASE 2: Bulk Feature Extraction (Hybrid)")
    print("=" * 60 + "\n")

    schema = load_schema(schema_path)
    regex_feats, llm_feats = split_features_by_lane(schema)
    llm_feature_names = build_llm_feature_list(llm_feats)
    print(f"Regex-extractable: {len(regex_feats)}")
    print(f"LLM-extractable:   {len(llm_feats)}\n")

    df = pd.read_csv(csv_path)
    df["truncated_caption"] = df["truncated_caption"].fillna("")
    n = len(df)
    print(f"Loaded {n:,} records")

    print("Running fast lane...")
    rule_results = df["truncated_caption"].apply(extract_row_rules).apply(pd.Series)
    print(f"Fast lane done: {len(rule_results.columns)} rule features")

    if use_llm and llm_feature_names:
        print(f"Running slow lane with Qwen ({LLM_BATCH_SIZE}/batch)...")
        system_prompt = build_llm_system_prompt(llm_feature_names)
        llm_results_list = [None] * n
        total_batches = 0

        for label_name in tqdm(df["label_name"].unique(), desc="Processing labels"):
            safe_name = re.sub(r"[^\w\-]", "_", str(label_name))[:60]
            ckpt_path = CHECKPOINT_DIR / f"llm_{safe_name}.json"

            label_mask = df["label_name"] == label_name
            label_indices = df.index[label_mask].tolist()
            label_captions = df.loc[label_indices, "truncated_caption"].tolist()

            if ckpt_path.exists():
                with open(ckpt_path, "r", encoding="utf-8") as f:
                    label_llm = json.load(f)
                if len(label_llm) == len(label_indices):
                    for idx, result in zip(label_indices, label_llm):
                        llm_results_list[idx] = result
                    continue

            label_llm = []
            for b_start in range(0, len(label_captions), LLM_BATCH_SIZE):
                batch = label_captions[b_start:b_start + LLM_BATCH_SIZE]
                batch_results = extract_llm_features(batch, llm_feature_names, system_prompt)
                label_llm.extend(batch_results)
                total_batches += 1
                time.sleep(0.4)

            with open(ckpt_path, "w", encoding="utf-8") as f:
                json.dump(label_llm, f)

            for idx, result in zip(label_indices, label_llm):
                llm_results_list[idx] = result

        fallback = {k: 2 for k in llm_feature_names}
        for i in range(n):
            if llm_results_list[i] is None:
                llm_results_list[i] = fallback

        llm_df = pd.DataFrame(llm_results_list)
        print(f"Slow lane done: {len(llm_df.columns)} LLM features ({total_batches} calls)")
    else:
        print("Skipping slow lane")
        llm_df = pd.DataFrame([{k: 2 for k in llm_feature_names} for _ in range(n)])

    meta_cols = df[["image", "label_name", "disease_label"]].reset_index(drop=True)
    final_df = pd.concat([meta_cols, rule_results.reset_index(drop=True), llm_df.reset_index(drop=True)], axis=1)

    NON_BINARY_COLS = {"image", "label_name", "disease_label", "age_numeric", "age_group", "duration_bucket", "diagnosis_confidence", "lesion_count"}
    categorical_llm = {f["name"] for f in llm_feats if not f.get("is_binary", True)}
    non_binary = NON_BINARY_COLS | categorical_llm

    binary_cols = [c for c in final_df.columns if c not in non_binary]
    for col in binary_cols:
        final_df[col] = pd.to_numeric(final_df[col], errors="coerce").fillna(2).astype(int)

    final_df.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} | shape={final_df.shape}")
    return final_df



In [ ]:
# =====================================================
# PHASE 3: Feature Analysis
# =====================================================

SCIN_TO_CANONICAL = {
    "textures_raised_or_bumpy": "texture_raised",
    "textures_flat": "texture_flat",
    "textures_rough_or_flaky": "texture_rough_flaky",
    "textures_fluid_filled": "texture_fluid_filled",
    "condition_symptoms_itching": "symptom_itching",
    "condition_symptoms_burning": "symptom_burning",
    "condition_symptoms_pain": "symptom_pain",
    "condition_symptoms_bleeding": "symptom_bleeding",
    "condition_symptoms_increasing_size": "symptom_increasing_size",
    "condition_symptoms_darkening": "symptom_darkening",
    "condition_symptoms_bothersome_appearance": "symptom_bothersome_appearance",
    "other_symptoms_fever": "symptom_fever",
    "other_symptoms_chills": "symptom_chills",
    "other_symptoms_fatigue": "symptom_fatigue",
    "other_symptoms_joint_pain": "symptom_joint_pain",
    "other_symptoms_mouth_sores": "symptom_mouth_sores",
    "other_symptoms_shortness_of_breath": "symptom_shortness_of_breath",
    "body_parts_head_or_neck": "location_head_neck",
    "body_parts_arm": "location_arm",
    "body_parts_palm": "location_palm",
    "body_parts_back_of_hand": "location_back_of_hand",
    "body_parts_torso_front": "location_torso_front",
    "body_parts_torso_back": "location_torso_back",
    "body_parts_genitalia_or_groin": "location_genitalia_groin",
    "body_parts_buttocks": "location_buttocks",
    "body_parts_leg": "location_leg",
    "body_parts_foot_top_or_side": "location_foot_top_side",
    "body_parts_foot_sole": "location_foot_sole",
}


def analyse_feature_coverage(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in feature_cols:
        vc = df[col].value_counts()
        n = len(df)
        rows.append({
            "feature": col,
            "n_present": vc.get(1, 0),
            "n_absent": vc.get(0, 0),
            "n_unknown": vc.get(2, 0),
            "pct_present": round(100 * vc.get(1, 0) / n, 2),
            "pct_absent": round(100 * vc.get(0, 0) / n, 2),
            "pct_unknown": round(100 * vc.get(2, 0) / n, 2),
            "informativeness": round(100 * (vc.get(1, 0) + vc.get(0, 0)) / n, 2),
        })
    return pd.DataFrame(rows).sort_values("informativeness", ascending=False)


def compare_derm_vs_scin(derm_df: pd.DataFrame, scin_df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    rows = []
    for scin_col, canonical in SCIN_TO_CANONICAL.items():
        if canonical not in feature_cols or scin_col not in scin_df.columns:
            continue

        derm_info = (derm_df[canonical] != 2).mean() * 100
        s_col = scin_df[scin_col]
        if s_col.dtype == object:
            scin_avail = (s_col.notna() & (s_col.astype(str).str.upper() == "YES")).mean() * 100
        else:
            scin_avail = s_col.notna().mean() * 100

        rows.append({
            "canonical_feature": canonical,
            "scin_column": scin_col,
            "derm1m_pct_informed": round(derm_info, 2),
            "scin_pct_available": round(scin_avail, 2),
            "gap": round(derm_info - scin_avail, 2),
        })

    return pd.DataFrame(rows).sort_values("gap", ascending=False)


def compute_feature_importance(df: pd.DataFrame, feature_cols: list[str], label_col: str = "label_name") -> pd.DataFrame:
    le = LabelEncoder()
    y = le.fit_transform(df[label_col].astype(str))

    mi_scores = []
    for col in feature_cols:
        mask = df[col] != 2
        x_sub = df.loc[mask, col].values.reshape(-1, 1)
        y_sub = y[mask.values]

        if len(x_sub) < 100:
            mi = 0.0
        else:
            mi = mutual_info_classif(x_sub, y_sub, discrete_features=True)[0]

        mi_scores.append({
            "feature": col,
            "mutual_information": round(mi, 5),
            "n_informed": int(mask.sum()),
        })

    return pd.DataFrame(mi_scores).sort_values("mutual_information", ascending=False)


def compute_classwise_importance(df: pd.DataFrame, feature_cols: list[str], label_col: str = "label_name", top_features: int = 20) -> dict[str, pd.DataFrame]:
    results = {}
    labels = df[label_col].unique()

    for label in labels:
        in_class = df[df[label_col] == label]
        out_class = df[df[label_col] != label]
        rows = []
        for col in feature_cols:
            in_pct = (in_class[col] == 1).sum() / max(len(in_class), 1)
            out_pct = (out_class[col] == 1).sum() / max(len(out_class), 1)
            or_val = (((in_pct + 1e-6) / (1 - in_pct + 1e-6)) / ((out_pct + 1e-6) / (1 - out_pct + 1e-6)))
            rows.append({
                "feature": col,
                "in_class_pct": round(in_pct * 100, 2),
                "out_class_pct": round(out_pct * 100, 2),
                "odds_ratio": round(or_val, 3),
            })
        results[label] = pd.DataFrame(rows).sort_values("odds_ratio", ascending=False).head(top_features)

    return results


FEATURE_TO_QUESTION = {
    "symptom_itching": "Does the rash itch?",
    "symptom_burning": "Does the affected area feel burning or stinging?",
    "symptom_pain": "Is the area painful or tender?",
    "symptom_bleeding": "Has the lesion bled?",
    "symptom_fever": "Have you had fever recently?",
    "symptom_joint_pain": "Any joint pain or swelling?",
    "symptom_fatigue": "Have you felt unusually fatigued?",
    "symptom_mouth_sores": "Do you have mouth sores or ulcers?",
    "symptom_increasing_size": "Has the lesion been growing or spreading?",
    "symptom_darkening": "Has the area been darkening over time?",
    "symptom_shortness_of_breath": "Any shortness of breath?",
    "symptom_chills": "Any chills or rigors?",
    "symptom_bothersome_appearance": "Is appearance bothersome?",
    "texture_raised": "Is the lesion raised or bumpy?",
    "texture_flat": "Is the lesion flat?",
    "texture_rough_flaky": "Is the area rough, scaly, or flaky?",
    "texture_fluid_filled": "Any blisters or fluid-filled bumps?",
    "texture_ulcerated": "Any open wound, ulcer, or crusting?",
    "onset_sudden": "Did it appear suddenly?",
    "trigger_identified": "Did anything trigger/worsen it (sun, meds, food)?",
    "family_history": "Family history of similar skin condition?",
    "recurrence": "Have you had this before?",
    "immunocompromised": "Do you have immune-related condition?",
    "associated_disease": "Any other diagnosed medical conditions?",
    "location_widespread": "Is rash affecting multiple body areas?",
    "location_face": "Is rash on face?",
    "location_trunk": "Is rash on chest/abdomen/back?",
    "location_genitalia_groin": "Is rash in genital/groin area?",
    "treatment_mentioned": "Have you tried any treatment?",
    "trigger_type": "What triggered/worsened the rash?",
}


def generate_questionnaire_for_cluster(disease_cluster: list[str], classwise_importance: dict[str, pd.DataFrame], top_n: int = 10) -> list[dict]:
    feature_scores: dict[str, list[float]] = {}
    for disease in disease_cluster:
        if disease not in classwise_importance:
            continue
        for _, row in classwise_importance[disease].iterrows():
            feat = row["feature"]
            feature_scores.setdefault(feat, []).append(row["odds_ratio"])

    ranked = sorted([(feat, np.mean(scores)) for feat, scores in feature_scores.items()], key=lambda x: -x[1])[:top_n]

    questionnaire = []
    for feat, score in ranked:
        questionnaire.append({
            "feature": feat,
            "discriminatory_score": round(score, 3),
            "question": FEATURE_TO_QUESTION.get(feat, f"Does the patient show signs of {feat.replace('_', ' ')}?"),
        })
    return questionnaire


def plot_coverage(coverage_df: pd.DataFrame, top_n: int = 40):
    top = coverage_df.head(top_n)
    fig, ax = plt.subplots(figsize=(12, 8))
    x = range(len(top))
    ax.bar(x, top["pct_present"], label="Present (1)", color="#2196F3")
    ax.bar(x, top["pct_absent"], bottom=top["pct_present"], label="Absent (0)", color="#90CAF9")
    ax.bar(x, top["pct_unknown"], bottom=top["pct_present"] + top["pct_absent"], label="Unknown (2)", color="#E0E0E0")
    ax.set_xticks(list(x))
    ax.set_xticklabels(top["feature"], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("% of records")
    ax.set_title(f"Feature Coverage in Derm-1M (top {top_n} most informative)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "feature_coverage.png", dpi=150)
    plt.close()


def plot_derm_vs_scin_comparison(cmp_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = range(len(cmp_df))
    ax.barh(list(y_pos), cmp_df["derm1m_pct_informed"], label="Derm-1M informed%", color="#1976D2")
    ax.barh(list(y_pos), cmp_df["scin_pct_available"], label="SCIN available%", color="#EF5350", alpha=0.7)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(cmp_df["canonical_feature"], fontsize=9)
    ax.set_xlabel("% of records with feature information")
    ax.set_title("Feature Informativeness: Derm-1M vs SCIN")
    ax.legend()
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "derm_vs_scin_comparison.png", dpi=150)
    plt.close()


def run_phase3(derm_features_csv: str = PHASE2_OUT, scin_csv: str = SCIN_CSV):
    print("=== Phase 3: Feature Analysis ===\n")

    derm_df = pd.read_csv(derm_features_csv)
    scin_df = pd.read_csv(scin_csv)

    META_COLS = {"image", "label_name", "disease_label", "age_numeric", "age_group", "duration_bucket", "diagnosis_confidence", "lesion_count"}
    feature_cols = [c for c in derm_df.columns if c not in META_COLS and derm_df[c].isin([0, 1, 2]).mean() > 0.8]

    coverage = analyse_feature_coverage(derm_df, feature_cols)
    coverage.to_csv(ANALYSIS_DIR / "feature_coverage.csv", index=False)
    plot_coverage(coverage)

    cmp_df = compare_derm_vs_scin(derm_df, scin_df, feature_cols)
    cmp_df.to_csv(ANALYSIS_DIR / "derm_vs_scin_comparison.csv", index=False)
    plot_derm_vs_scin_comparison(cmp_df)

    mi_df = compute_feature_importance(derm_df, feature_cols, label_col="label_name")
    mi_df.to_csv(ANALYSIS_DIR / "feature_importance_global.csv", index=False)

    cw_importance = compute_classwise_importance(derm_df, feature_cols, label_col="label_name")
    cw_dir = ANALYSIS_DIR / "classwise_importance"
    cw_dir.mkdir(exist_ok=True)
    for disease, df_imp in cw_importance.items():
        safe_name = disease.replace("/", "_").replace(" ", "_")[:60]
        df_imp.to_csv(cw_dir / f"{safe_name}.csv", index=False)

    example_cluster = ["eczema", "dermatitis", "psoriasis"]
    questionnaire = generate_questionnaire_for_cluster(example_cluster, cw_importance)
    with open(ANALYSIS_DIR / "questionnaire_eczema_cluster.json", "w", encoding="utf-8") as f:
        json.dump({"cluster": example_cluster, "questions": questionnaire}, f, indent=2)

    scin_canonical_set = set(SCIN_TO_CANONICAL.values())
    derm_only_features = [c for c in feature_cols if c not in scin_canonical_set]
    coverage[coverage["feature"].isin(derm_only_features)].to_csv(ANALYSIS_DIR / "derm_only_features.csv", index=False)

    print(f"Saved Phase 3 outputs to: {ANALYSIS_DIR}")
    return {
        "coverage": coverage,
        "comparison": cmp_df,
        "mi": mi_df,
        "classwise": cw_importance,
    }



In [ ]:
# =====================================================
# PHASE 3b: Per-Disease Co-occurrence Analysis
# =====================================================

MIN_SUPPORT = 0.15
MIN_PAIR_SUPPORT = 0.10
TOP_FEATURES_PLOT = 20


def phi_coefficient(x: np.ndarray, y: np.ndarray) -> float:
    mask = (x != 2) & (y != 2)
    if mask.sum() < 10:
        return 0.0
    x_b = (x[mask] == 1).astype(int)
    y_b = (y[mask] == 1).astype(int)
    if x_b.std() == 0 or y_b.std() == 0:
        return 0.0
    r, _ = pearsonr(x_b, y_b)
    return round(float(r), 4)


def compute_disease_cooccurrence(df: pd.DataFrame, feature_cols: list[str], disease: str, label_col: str = "label_name") -> pd.DataFrame:
    subset = df[df[label_col] == disease][feature_cols]
    n_cols = subset.shape[1]
    phi_matrix = np.zeros((n_cols, n_cols))

    for i, f1 in enumerate(feature_cols):
        for j, f2 in enumerate(feature_cols):
            if i == j:
                phi_matrix[i, j] = 1.0
            elif i < j:
                phi = phi_coefficient(subset[f1].values, subset[f2].values)
                phi_matrix[i, j] = phi
                phi_matrix[j, i] = phi

    return pd.DataFrame(phi_matrix, index=feature_cols, columns=feature_cols)


def extract_diagnostic_signature(
    df: pd.DataFrame,
    feature_cols: list[str],
    disease: str,
    label_col: str = "label_name",
    min_support: float = MIN_SUPPORT,
    min_pair_support: float = MIN_PAIR_SUPPORT,
    top_k_singles: int = 15,
) -> dict:
    subset = df[df[label_col] == disease]
    n = len(subset)
    if n < 20:
        return {"disease": disease, "n_cases": n, "singles": [], "pairs": [], "triplets": []}

    singles = []
    for f in feature_cols:
        col = subset[f]
        n_informed = (col != 2).sum()
        n_present = (col == 1).sum()
        support = n_present / n if n > 0 else 0
        if support >= min_support:
            singles.append({
                "feature": f,
                "support": round(support, 3),
                "n_present": int(n_present),
                "n_informed": int(n_informed),
                "pct_of_informed": round(n_present / n_informed, 3) if n_informed > 0 else 0,
            })
    singles.sort(key=lambda x: -x["support"])

    top_feats = [s["feature"] for s in singles[:top_k_singles]]
    pairs = []
    for f1, f2 in combinations(top_feats, 2):
        mask = (subset[f1] != 2) & (subset[f2] != 2)
        n_both = ((subset.loc[mask, f1] == 1) & (subset.loc[mask, f2] == 1)).sum()
        support = n_both / n if n > 0 else 0
        phi = phi_coefficient(subset[f1].values, subset[f2].values)
        if support >= min_pair_support:
            pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "joint_support": round(support, 3),
                "n_both": int(n_both),
                "phi_coefficient": phi,
            })
    pairs.sort(key=lambda x: -x["joint_support"])

    top10 = [s["feature"] for s in singles[:10]]
    triplets = []
    for f1, f2, f3 in combinations(top10, 3):
        mask = (subset[f1] != 2) & (subset[f2] != 2) & (subset[f3] != 2)
        n_all3 = ((subset.loc[mask, f1] == 1) & (subset.loc[mask, f2] == 1) & (subset.loc[mask, f3] == 1)).sum()
        support = n_all3 / n if n > 0 else 0
        if support >= min_pair_support:
            triplets.append({
                "features": [f1, f2, f3],
                "joint_support": round(support, 3),
                "n_all_present": int(n_all3),
            })
    triplets.sort(key=lambda x: -x["joint_support"])

    return {"disease": disease, "n_cases": n, "singles": singles, "pairs": pairs[:30], "triplets": triplets[:20]}


def build_scin_feature_matrix(scin_df: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame(index=scin_df.index)

    for scin_col, canonical in SCIN_TO_CANONICAL.items():
        if scin_col not in scin_df.columns:
            continue
        col = scin_df[scin_col]
        if col.dtype == object:
            vals = col.astype(str).str.upper().str.strip()
            encoded = vals.map({"YES": 1}).fillna(vals.map({"NO": 0}).fillna(2)).astype(int)
        else:
            encoded = col.notna().astype(int) * 1

        if canonical in result.columns:
            result[canonical] = result[canonical].combine(
                encoded,
                lambda a, b: 1 if (a == 1 or b == 1) else (2 if (a == 2 and b == 2) else 0),
            )
        else:
            result[canonical] = encoded

    return result


def signature_completeness_in_scin(signature: dict, scin_features: pd.DataFrame, canonical_available: set[str]) -> dict:
    disease = signature["disease"]
    scin_sub = scin_features
    n_scin = max(len(scin_sub), 1)

    singles_coverage = []
    for s in signature["singles"]:
        feat = s["feature"]
        available = feat in canonical_available
        if available and feat in scin_sub.columns:
            pct_informed = (scin_sub[feat] != 2).mean()
            pct_present = (scin_sub[feat] == 1).mean()
        else:
            pct_informed, pct_present = 0.0, 0.0
        singles_coverage.append({
            "feature": feat,
            "derm1m_support": s["support"],
            "scin_col_available": available,
            "scin_pct_informed": round(pct_informed, 3),
            "scin_pct_present": round(pct_present, 3),
            "gap": round(s["support"] - pct_present, 3),
        })

    pairs_coverage = []
    for p in signature["pairs"]:
        f1, f2 = p["feature_1"], p["feature_2"]
        both_avail = (f1 in canonical_available) and (f2 in canonical_available)
        if both_avail and f1 in scin_sub.columns and f2 in scin_sub.columns:
            mask = (scin_sub[f1] != 2) & (scin_sub[f2] != 2)
            n_both = ((scin_sub.loc[mask, f1] == 1) & (scin_sub.loc[mask, f2] == 1)).sum()
            scin_joint = n_both / n_scin
            phi_scin = phi_coefficient(scin_sub[f1].values, scin_sub[f2].values)
        else:
            scin_joint, phi_scin = 0.0, 0.0
        pairs_coverage.append({
            "feature_1": f1,
            "feature_2": f2,
            "derm1m_joint_support": p["joint_support"],
            "derm1m_phi": p["phi_coefficient"],
            "both_cols_in_scin": both_avail,
            "scin_joint_support": round(scin_joint, 3),
            "scin_phi": round(phi_scin, 3),
            "joint_support_gap": round(p["joint_support"] - scin_joint, 3),
            "phi_gap": round(p["phi_coefficient"] - phi_scin, 3),
        })

    n_singles = len(singles_coverage)
    n_covered = sum(1 for s in singles_coverage if s["scin_pct_informed"] >= 0.05)
    coverage_score = n_covered / n_singles if n_singles > 0 else 0

    n_pairs = len(pairs_coverage)
    n_intact = sum(1 for p in pairs_coverage if p["derm1m_phi"] > 0 and p["scin_phi"] >= 0.5 * p["derm1m_phi"])
    pair_integrity = n_intact / n_pairs if n_pairs > 0 else 0

    return {
        "disease": disease,
        "n_derm1m_cases": signature["n_cases"],
        "n_scin_cases": n_scin,
        "singles_coverage": singles_coverage,
        "pairs_coverage": pairs_coverage,
        "summary": {
            "n_signature_features": n_singles,
            "n_features_in_scin": n_covered,
            "single_feature_coverage": round(coverage_score, 3),
            "n_signature_pairs": n_pairs,
            "n_pairs_intact_in_scin": n_intact,
            "pair_integrity_score": round(pair_integrity, 3),
            "signature_reproducibility": round((coverage_score + pair_integrity) / 2, 3),
        },
    }


def confusion_aware_gap(signatures: dict[str, dict], confusion_pairs: list[tuple[str, str]]) -> list[dict]:
    results = []
    for label_a, label_b in confusion_pairs:
        if label_a not in signatures or label_b not in signatures:
            continue

        sig_a, sig_b = signatures[label_a], signatures[label_b]
        a_singles = {s["feature"]: s["support"] for s in sig_a["singles"]}
        b_singles = {s["feature"]: s["support"] for s in sig_b["singles"]}

        discriminating = []
        for feat, a_sup in a_singles.items():
            b_sup = b_singles.get(feat, 0)
            lift = a_sup - b_sup
            if lift >= 0.10:
                discriminating.append({
                    "feature": feat,
                    "support_in_A": a_sup,
                    "support_in_B": b_sup,
                    "discriminating_lift": round(lift, 3),
                })
        discriminating.sort(key=lambda x: -x["discriminating_lift"])

        a_pairs = {(p["feature_1"], p["feature_2"]): p for p in sig_a["pairs"]}
        b_pair_keys = {(p["feature_1"], p["feature_2"]) for p in sig_b["pairs"]}

        disc_pairs = []
        for (f1, f2), p in a_pairs.items():
            if (f1, f2) not in b_pair_keys:
                disc_pairs.append({
                    "feature_1": f1,
                    "feature_2": f2,
                    "joint_support_in_A": p["joint_support"],
                    "phi_in_A": p["phi_coefficient"],
                    "present_in_B_signature": False,
                })

        results.append({
            "true_label": label_a,
            "confused_with": label_b,
            "discriminating_singles": discriminating[:10],
            "discriminating_pairs": disc_pairs[:10],
            "n_discriminators": len(discriminating),
        })

    return results


def plot_cooccurrence_heatmap(phi_df: pd.DataFrame, disease: str, top_n: int = TOP_FEATURES_PLOT):
    max_phi = phi_df.abs().replace(1.0, 0).max(axis=1)
    top_feats = max_phi.nlargest(top_n).index.tolist()
    sub = phi_df.loc[top_feats, top_feats]

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.eye(len(top_feats), dtype=bool)
    sns.heatmap(sub, mask=mask, cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5, annot=True, fmt=".2f", annot_kws={"size": 7}, linewidths=0.3, ax=ax, square=True)
    ax.set_title(f"Feature Co-occurrence (Phi) - {disease}", fontsize=12, pad=12)
    plt.tight_layout()
    safe = disease.replace("/", "_").replace(" ", "_")[:50]
    path = COOC_DIR / f"cooccurrence_{safe}.png"
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    return path


def plot_signature_gap(completeness: dict, top_n: int = 15):
    singles = completeness["singles_coverage"][:top_n]
    if not singles:
        return

    feats = [s["feature"].replace("_", " ") for s in singles]
    derm = [s["derm1m_support"] for s in singles]
    scin = [s["scin_pct_present"] for s in singles]

    x = np.arange(len(feats))
    w = 0.35
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w / 2, derm, w, label="Derm-1M support", color="#1565C0")
    ax.bar(x + w / 2, scin, w, label="SCIN present%", color="#EF5350", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(feats, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("Proportion of class records")

    disease = completeness["disease"]
    ax.set_title(
        f"Signature Feature Gap - {disease}\n"
        f"Coverage: {completeness['summary']['single_feature_coverage']:.0%} | "
        f"Pair Integrity: {completeness['summary']['pair_integrity_score']:.0%} | "
        f"Reproducibility: {completeness['summary']['signature_reproducibility']:.0%}"
    )
    ax.legend()
    plt.tight_layout()
    safe = disease.replace("/", "_").replace(" ", "_")[:50]
    path = COOC_DIR / f"signature_gap_{safe}.png"
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    return path


def plot_reproducibility_summary(all_completeness: list[dict]):
    rows = [(c["disease"], c["summary"]["signature_reproducibility"], c["summary"]["single_feature_coverage"], c["summary"]["pair_integrity_score"]) for c in all_completeness]
    rows.sort(key=lambda x: x[1])

    diseases = [r[0] for r in rows]
    reprod = [r[1] for r in rows]
    single = [r[2] for r in rows]
    pair_int = [r[3] for r in rows]

    fig, ax = plt.subplots(figsize=(14, max(6, len(diseases) * 0.3)))
    y = np.arange(len(diseases))
    ax.barh(y, reprod, color="#1565C0", label="Signature Reproducibility")
    ax.barh(y - 0.25, single, height=0.2, color="#42A5F5", alpha=0.8, label="Single-feature Coverage")
    ax.barh(y + 0.25, pair_int, height=0.2, color="#EF9A9A", alpha=0.8, label="Pair Integrity")
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="0.5 threshold")
    ax.set_yticks(y)
    ax.set_yticklabels(diseases, fontsize=8)
    ax.set_xlabel("Score (0-1)")
    ax.set_title("SCIN Signature Reproducibility per Disease Class")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    path = COOC_DIR / "reproducibility_summary.png"
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    return path


def run_phase3b(derm_features_csv: str = PHASE2_OUT, scin_csv: str = SCIN_CSV):
    print("=== Phase 3b: Co-occurrence Analysis ===\n")

    derm_df = pd.read_csv(derm_features_csv)
    scin_raw = pd.read_csv(scin_csv)

    META_COLS = {"image", "label_name", "disease_label", "age_numeric", "age_group", "duration_bucket", "diagnosis_confidence", "lesion_count"}
    feature_cols = [c for c in derm_df.columns if c not in META_COLS and derm_df[c].isin([0, 1, 2]).mean() > 0.8]

    diseases = derm_df["label_name"].value_counts()
    print(f"Derm records={len(derm_df):,}, features={len(feature_cols)}, classes={len(diseases)}")

    scin_features = build_scin_feature_matrix(scin_raw)
    canonical_in_scin = set(scin_features.columns)

    all_signatures = {}
    for disease in diseases.index:
        all_signatures[disease] = extract_diagnostic_signature(derm_df, feature_cols, disease, label_col="label_name")

    with open(COOC_DIR / "all_signatures.json", "w", encoding="utf-8") as f:
        json.dump(all_signatures, f, indent=2)

    all_completeness = []
    for disease, sig in all_signatures.items():
        comp = signature_completeness_in_scin(sig, scin_features, canonical_in_scin)
        all_completeness.append(comp)
        plot_signature_gap(comp)

    summary_rows = [c["summary"] | {"disease": c["disease"], "n_derm1m": c["n_derm1m_cases"], "n_scin": c["n_scin_cases"]} for c in all_completeness]
    summary_df = pd.DataFrame(summary_rows).sort_values("signature_reproducibility")
    summary_df.to_csv(COOC_DIR / "signature_completeness_summary.csv", index=False)
    plot_reproducibility_summary(all_completeness)

    worst_diseases = summary_df.head(10)["disease"].tolist()
    for disease in worst_diseases:
        phi_df = compute_disease_cooccurrence(derm_df, feature_cols, disease, label_col="label_name")
        plot_cooccurrence_heatmap(phi_df, disease)

    confusion_pairs = [
        ("eczema", "dermatitis"),
        ("eczema", "psoriasis"),
        ("psoriasis", "dermatitis"),
        ("melanoma", "nevus (mole)"),
        ("basal cell carcinoma", "squamous cell carcinoma"),
        ("tinea (ringworm)", "eczema"),
        ("urticaria (hives)", "dermatitis"),
        ("dermatitis", "eczema"),
        ("acne", "folliculitis (inflamed hair follicles)"),
        ("herpes simplex virus", "herpes zoster (shingles)"),
    ]
    conf_gaps = confusion_aware_gap(all_signatures, confusion_pairs)
    with open(COOC_DIR / "confusion_aware_gaps.json", "w", encoding="utf-8") as f:
        json.dump(conf_gaps, f, indent=2)

    print(f"Saved Phase 3b outputs to: {COOC_DIR}")
    return {
        "summary": summary_df,
        "signatures": all_signatures,
        "confusion_gaps": conf_gaps,
    }



## Run Phases

Run each phase separately (recommended for long jobs), or run all in sequence.



In [ ]:
# Phase 1
# schema = run_phase1(CSV_PATH, SCHEMA_OUT)

# Phase 2
# features_df = run_phase2(CSV_PATH, SCHEMA_OUT, PHASE2_OUT, use_llm=True)

# Phase 3
# phase3_outputs = run_phase3(PHASE2_OUT, SCIN_CSV)

# Phase 3b
# phase3b_outputs = run_phase3b(PHASE2_OUT, SCIN_CSV)

